# JSON-Dateien

## Was ist JSON?

JavaScript Object Notation ist ein einfaches textbasiertes Datenaustauschformat, lesbar für Menschen und Maschinen.


### Beispiel

<pre>{
  "measurements": [
    {
      "measurementId": "1234",
      "current_A": 15.6,
      "voltage_V": 230,
      "timestamp": "2025-01-10T10:30:00Z"
    },
    {
      "measurementId": "1235",
      "current_A": 12.4,
      "voltage_V": null,
      "timestamp": "2025-01-10T10:31:00Z"
    }
  ],
  "validated": false
}</pre>


JSON besteht aus *Objects* und *Arrays*.


- Ein *Object* ist ein Set von Name:Wert Paaren.
- Ein *Array* ist eine geordnete Liste von Werten.
- Ein *Wert* kann sein:
    - Object
    - Array
    - String
    - Zahl
    - true
    - false
    - null
- Ein *Name* ist immer ein String.

Details: https://www.json.org/json-de.html

Als Pythonobjekt sehen die gleichen Daten so aus:

In [ ]:
data = {
  "measurements": [
    {
      "measurementId": "1234",
      "current_A": 15.6,
      "voltage_V": 230,
      "timestamp": "2025-01-10T10:30:00Z"
    },
    {
      "measurementId": "1235",
      "current_A": 12.4,
      "voltage_V": None,
      "timestamp": "2025-01-10T10:31:00Z"
    }
  ],
  "validated": False
}

## JSON schreiben und lesen

Eine JSON-Bibliothek ist in der Python Standard Library enthalten: https://docs.python.org/3/library/json.html

In [ ]:
import json  # standard library

with open("measurements.json", mode="wt") as f:
    json.dump(data, f, indent=2)

In [ ]:
with open("measurements.json", mode="rt") as f:
    parsed_data = json.load(f)

parsed_data

In [ ]:
parsed_data == data

Man muss JSON nicht direkt in ein File schreiben:

In [ ]:
# .dumps statt .dump erzeugt einen str
json_str_data = json.dumps(data)
json_str_data

### Was ist mit Zeitstempeln?

Alle JSON-Typen haben direkte Entsprechungen in Python:

| JSON   | Python       |
|--------|--------------|
| object | dict         |
| array  | list, tuple  |
| string | str          |
| number | int, float   |
| true   | True         |
| false  | False        |
| null   | None         |



Aber was passiert mit Pythonobjekten, die kein Äquivalent im JSON Standard haben?

In [ ]:
from datetime import datetime, UTC

measurements = [
    {
      "measurementId": "1236",
      "current_A": 15.6,
      "voltage_V": 230,
      "timestamp": datetime(2025, 1, 30, 14, 15, tzinfo=UTC)
    }
]

In [ ]:
json.dumps(measurements)

#### Angepasste Serialisierung

In [ ]:
# Fix 1 - alles vorher konvertieren
import copy

serializable_measurements = copy.deepcopy(measurements)

for m in serializable_measurements:
    m["timestamp"] = m["timestamp"].isoformat()

json.dumps(serializable_measurements)

In [ ]:
# Fix 2 - custom serialization

def serialize_dt(non_json_value):
    if isinstance(non_json_value, datetime):
        return non_json_value.isoformat()
    else:
        raise TypeError(f'unexpected: {non_json_value}')

json.dumps(
    measurements, 
    default=serialize_dt  # Achtung: kein ()
)

#### Angepasstes Parsing

In [ ]:
json_measurement = '{"measurementId": "1236", "current_A": 15.6, "voltage_V": 230, "timestamp": "2025-01-30T14:15:00Z"}'

In [ ]:
json.loads(json_measurement)

Der Timestamp ist ein String - meistens nicht so nützlich.

In [ ]:
# Custom Funktion fürs Verarbeiten des JSON-Objekts.
def parse_timestamps(parsed_json_obj):
    if 'timestamp' in parsed_json_obj:
        parsed_json_obj['timestamp'] = datetime.fromisoformat(parsed_json_obj['timestamp'])
    return parsed_json_obj



json.loads(json_measurement, object_hook=parse_timestamps)

#### Ausblick: Pydantic

Das Einlesen von Daten in Pythonobjekte kann schnell sehr komplex werden.

Das beste Paket für Parsing und Validierung von Daten in JSON und ähnlichen Formaten ist [**Pydantic**](https://docs.pydantic.dev/latest/):

In [ ]:
import pydantic
from datetime import datetime

# Definiere ein Ziel-Datenmodell:

class Measurement(pydantic.BaseModel):
    # Gleiche Syntax für Attribute wie bei dataclass.
    measurementId: int
    current_A: float
    voltage_V: float
    timestamp: datetime

class Measurements(pydantic.BaseModel):
    measurements: list[Measurement]


In [ ]:
# Parsing direkt von JSON, mit den verschachtelten Klassen.
json_measurements = '{"measurements": [{"measurementId": "1236", "current_A": 15.6, "voltage_V": 230, "timestamp": "2025-01-30T14:15:00Z"}]}'

pydantic_measurements = Measurements.model_validate_json(json_measurements)
pydantic_measurements

In [ ]:
# Serialisierung ist auch einfach:
pydantic_measurements.model_dump_json()

In [ ]:
# In ein File mit:
with open('measurements.json', 'wt') as f:
    f.write(pydantic_measurements.model_dump_json(indent=2))

## Übung

Hier im Ordner `/data` liegt ein File `raw_measurements.json`. Im "measurements" Object liegt ein Array von Messungen, ähnlich wie oben. 

1. Schreibe Code, der das File in ein Pythonobjekt einliest.
2. Behalte von den Messungen nur die mit `validated: true` (Tipp: for-Loop, if-Abfrage).
3. Berechne für jede behaltene Messung die Leistung in kW und füge sie hinzu (W = V*A).
4. Schreibe alles wieder in ein "clean" File raus.

**Bonus**: Berechne noch die Zeitabstände zwischen den Messungen (Tipp: Zeitstempel in datetime konvertieren, dann subtrahieren). 



.
 
.
 
.
 
.
 
.
 
.
 
.
 
.
 
.
 
.
 
.
 
.

#### Mögliche Lösung

In [ ]:
import json
from pathlib import Path

# Pfad zum Input
input_path = Path('.').parent / "data" / "raw_measurements.json"

# Unbearbeitete Daten
with input_path.open("rt", encoding="utf-8") as f:
    data = json.load(f)

# Bearbeite eine Messung nach der anderen:
clean_measurements = []
for m in data["measurements"]:
    # Behalte nur validierte Messungen:
    if m["validated"]:
        # Berechnung nur wenn A und V nicht None sind:
        if m["current_A"] is not None and m["voltage_V"] is not None:
            power_kW = (m["current_A"] * m["voltage_V"]) / 1000
            # Runde auf 2 Nachkommastellen:
            m["power_kW"] = round(power_kW, 2)
        else:
            # Wenn Berechnung nicht möglich:
            m["power_kW"] = None
        # Lösche Feld:
        del m["validated"]

    clean_measurements.append(m)

# Ersetze Messungen mit sauberen:
data["measurements"] = clean_measurements

# Schreibe raus:
output_path = input_path.parent / 'cleaned_measurements.json'
with output_path.open("wt") as f:
    json.dump(data, f, indent=2)
